In [1]:
import numpy as np 
from scipy.io import loadmat
import librosa
import soundfile as sf
import glob 
from IPython.display import Audio 

In [2]:
sources = 'Source/*.wav'
surround_sources = 'Source/Surround Quick Test/*.wav'
hrtfs = 'HRTF/*.wav'
brirs = 'BRIR/*.wav'
rirs = 'RIR/*.wav'
_SOURCES = glob.glob(sources) 
_QUICK = glob.glob(surround_sources) 
_HRIR = glob.glob(hrtfs)
_RIR = glob.glob(rirs)
_NYU = glob.glob(brirs)

In [3]:
#Stereo sample
stereo, fs = sf.read(_SOURCES[1])
Audio(stereo.transpose(), rate = fs)

PROCESSOR WITH SURROUND/STEREO INPUTS:

Azimuth degrees for correct BRIR selection (when navigating HRTF datasets): 

Surround 5.1: middle = 0, left = 30, right = 330, left_surround = 110, right_surround = 250. Elevation = 0.
Stereo: left = 45, right = 315.


To deactivate a channel in "binauralsurround" mode just put an arbitrary string as argument, like "None".

In [4]:
# Surround/Stereo Binaural Processor  

class combinedbinauralprocessor(): 
    class surround(): #Pass BRIRs with corresponding azimuth degrees here:
        def __init__(self, f_middle, f_right, f_left, f_surround_left, f_surround_right, f_sub, ir_middle, ir_right, ir_left, ir_sr, ir_sl, ir_sub, compensation):
            self.f_middle = f_middle
            self.f_right = f_right
            self.f_left = f_left
            self.f_surround_left = f_surround_left
            self.f_surround_right = f_surround_right
            self.f_sub = f_sub
            self.ir_middle = ir_middle
            self.ir_right = ir_right
            self.ir_left = ir_left 
            self.ir_sl = ir_sl
            self.ir_sr = ir_sr 
            self.ir_sub = ir_sub
            self.compensation = compensation
        #Pass surround mono channel inputs here: To deactivate channel pass an arbitrary value
        def binauralsurround(self, middle, right, left, right_surround, left_surround, sub, HRTForBRIR = 'HRTF'):
            K = np.max([len(self.f_middle), len(self.f_right), len(self.f_left), len(self.f_surround_left), len(self.f_surround_right), len(self.f_sub), len(self.ir_middle), len(self.ir_right), len(self.ir_left), len(self.ir_sl), len(self.ir_sr), len(self.ir_sub)])
            N = np.max([len(middle), len(left), len(right), len(left_surround), len(right_surround)]) 
            leftchannel = np.zeros(N + K - 1)
            rightchannel = np.zeros(N + K - 1)
            output = np.array([leftchannel, rightchannel])
            comp = np.zeros(N + K - 1) 
            comp[0: len(self.compensation)] = self.compensation
            compfft = np.fft.fft(comp) #compensation filter 
            holder = np.zeros(N + K - 1) #filter buffer
            padded = np.zeros(N + K - 1) #signal buffer
            if HRTForBRIR == 'BRIR':
                if type(self.ir_middle) and type(self.ir_right) and type(self.ir_left) and type(self.ir_sr) and type(self.ir_sl) == str:
                    if type(middle) == np.ndarray:
                        for i in range(0, 2): #middle channel hrtf convolution for both ears
                            holder[0: len(self.f_middle[:, i])] += self.f_middle[:, i]
                            fftfilter = np.fft.fft(holder)
                            padded[0: len(middle)] = middle 
                            convolve = fftfilter * np.fft.fft(padded) * compfft
                            output[i, :] += np.real(np.fft.ifft(convolve))
                            holder *= 0
                            padded *= 0
                    if type(right) == np.ndarray:
                        for i in range(0, 2): #right channel hrtf convolution for both ears
                            holder[0: len(self.f_right[:, i])] += self.f_right[:, i]
                            fftfilter = np.fft.fft(holder)
                            padded[0: len(right)] = right
                            convolve = fftfilter * np.fft.fft(padded) * compfft
                            output[i, :] += np.real(np.fft.ifft(convolve))
                            holder *= 0 
                            padded *= 0
                    if type(left) == np.ndarray:
                        for i in range(0, 2): #left channel hrtf convolution for both ears
                            holder[0: len(self.f_left[:, i])] += self.f_left[:, i]
                            fftfilter = np.fft.fft(holder)
                            padded[0: len(left)] = left
                            convolve = fftfilter * np.fft.fft(padded) * compfft
                            output[i, :] += np.real(np.fft.ifft(convolve))
                            holder *= 0 
                            padded *= 0
                    if type(left_surround) == np.ndarray:
                        for i in range(0, 2): #surround left channel hrtf convolution for both ears
                            holder[0: len(self.f_surround_left)] += self.f_surround_left[:, i]
                            fftfilter = np.fft.fft(holder)
                            padded[0: len(left_surround)] = left_surround
                            convolve = fftfilter * np.fft.fft(padded) * compfft
                            output[i, :] += np.real(np.fft.ifft(convolve))
                            holder *= 0 
                            padded *= 0 
                    if type(right_surround) == np.ndarray:
                        for i in range(0, 2): #surround right channel hrtf convolution for both ears
                            holder[0: len(self.f_surround_right[:, i])] += self.f_surround_right[:, i]
                            fftfilter = np.fft.fft(holder)
                            padded[0: len(right_surround)] = right_surround
                            convolve = fftfilter * np.fft.fft(padded) * compfft
                            output[i, :] += np.real(np.fft.ifft(convolve))
                            holder *= 0 
                            padded *= 0
                    if type(sub) == np.ndarray:
                        for i in range(0, 2): #surround right channel hrtf convolution for both ears
                            holder[0: len(self.f_sub[:, i])] += self.f_sub[:, i]
                            fftfilter = np.fft.fft(holder)
                            padded[0: len(sub)] = sub
                            convolve = fftfilter * np.fft.fft(padded) * compfft 
                            output[i, :] += np.real(np.fft.ifft(convolve))
                            holder *= 0 
                            padded *= 0
                else: 
                    print('Room impulse responses should be omitted for BRIR convolution')
                    return 
                
            elif HRTForBRIR == 'HRTF':
                rir = np.zeros(N + K - 1)
                if type(middle) == np.ndarray:
                    for i in range(0, 2): #middle channel hrtf convolution for both ears
                        holder[0: len(self.f_middle[:, i])] += self.f_middle[:, i]
                        fftfilter = np.fft.fft(holder)
                        rir[0: len(self.ir_middle)] = self.ir_middle
                        padded[0: len(middle)] = middle 
                        convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0
                        padded *= 0
                        rir *= 0 
                if type(right) == np.ndarray:
                    for i in range(0, 2): #right channel hrtf convolution for both ears
                        holder[0: len(self.f_right[:, i])] += self.f_right[:, i]
                        fftfilter = np.fft.fft(holder)
                        rir[0: len(self.ir_right)] = self.ir_right
                        padded[0: len(right)] = right
                        convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0 
                        padded *= 0
                        rir *= 0 
                if type(left) == np.ndarray:
                    for i in range(0, 2): #left channel hrtf convolution for both ears
                        holder[0: len(self.f_left[:, i])] += self.f_left[:, i]
                        fftfilter = np.fft.fft(holder)
                        rir[0: len(self.ir_left)] = self.ir_left
                        padded[0: len(left)] = left
                        convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0 
                        padded *= 0
                        rir *= 0 
                if type(left_surround) == np.ndarray:
                    for i in range(0, 2): #surround left channel hrtf convolution for both ears
                        holder[0: len(self.f_surround_left)] += self.f_surround_left[:, i]
                        fftfilter = np.fft.fft(holder)
                        rir[0: len(self.ir_sl)] = self.ir_sl
                        padded[0: len(left_surround)] = left_surround
                        convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0 
                        padded *= 0 
                        rir *= 0 
                if type(right_surround) == np.ndarray:
                    for i in range(0, 2): #surround right channel hrtf convolution for both ears
                        holder[0: len(self.f_surround_right[:, i])] += self.f_surround_right[:, i]
                        fftfilter = np.fft.fft(holder)
                        rir[0: len(self.ir_sr)] = self.ir_sr
                        padded[0: len(right_surround)] = right_surround
                        convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0 
                        padded *= 0
                        rir *= 0 
                if type(sub) == np.ndarray:
                    for i in range(0, 2): #surround right channel hrtf convolution for both ears
                        holder[0: len(self.f_sub[:, i])] += self.f_sub[:, i]
                        fftfilter = np.fft.fft(holder)
                        rir[0: len(self.ir_sub)] = self.ir_sub
                        padded[0: len(sub)] = sub
                        convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0 
                        padded *= 0
                        rir *= 0 
            return output
    class stereomode():
        def __init__(self, f_right_stereo, f_left_stereo, ir_left, ir_right, compensation):
            self.f_left_stereo = f_left_stereo
            self.f_right_stereo = f_right_stereo
            self.compensation = compensation
            self.ir_left = ir_left
            self.ir_right = ir_right
        #Pass stereo input here 
        def binaural(self, stereo, HRTForBRIR = 'HRTF'):
            K = np.max([len(self.f_left_stereo), len(self.f_right_stereo), len(self.ir_left), len(self.ir_right)])
            N = len(stereo)
            leftchannel = np.zeros(N + K - 1)
            rightchannel = np.zeros(N + K - 1)
            output = np.array([leftchannel, rightchannel])
            stereo_right = stereo[:, 1] 
            stereo_left = stereo[:, 0]
            comp = np.zeros(N + K - 1) 
            comp[0: len(self.compensation)] = self.compensation
            compfft = np.fft.fft(comp)
            holder = np.zeros(N + K - 1) #filter buffer
            padded = np.zeros(N + K - 1) #padded signal for full mode
            if HRTForBRIR == 'BRIR':
                if type(self.ir_left) and type(self.ir_right) == str:
                    for i in range(0, 2): #right channel hrtf convolution for both ears
                        holder[0: len(self.f_right_stereo[:, i])] += self.f_right_stereo[:, i]
                        fftfilter = np.fft.fft(holder)
                        padded[0: N] = stereo_right
                        convolve = fftfilter * np.fft.fft(padded) * compfft 
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0 
                        padded *= 0
                    for i in range(0, 2): #left channel hrtf convolution for both ears
                        holder[0: len(self.f_left_stereo[:, i])] += self.f_left_stereo[:, i]
                        fftfilter = np.fft.fft(holder)
                        padded[0: N] = stereo_left
                        convolve = fftfilter * np.fft.fft(padded) * compfft
                        output[i, :] += np.real(np.fft.ifft(convolve))
                        holder *= 0
                        padded *= 0
                else: 
                    print('Impulse responses should be omitted for BRIR convolution')
                    return 
            elif HRTForBRIR == 'HRTF':
                rir = np.zeros(N + K - 1)
                for i in range(0, 2): #right channel hrtf convolution for both ears
                    holder[0: len(self.f_right_stereo[:, i])] += self.f_right_stereo[:, i]
                    fftfilter = np.fft.fft(holder)
                    rir[0: len(self.ir_right)] = self.ir_right
                    padded[0: N] = stereo_right
                    convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                    output[i, :] += np.real(np.fft.ifft(convolve))
                    holder *= 0 
                    padded *= 0
                    rir *= 0 
                for i in range(0, 2): #left channel hrtf convolution for both ears
                    holder[0: len(self.f_left_stereo[:, i])] += self.f_left_stereo[:, i]
                    fftfilter = np.fft.fft(holder)
                    rir[0: len(self.ir_left)] = self.ir_left
                    padded[0: N] = stereo_left
                    convolve = fftfilter * np.fft.fft(padded) * compfft * np.fft.fft(rir)
                    output[i, :] += np.real(np.fft.ifft(convolve))
                    holder *= 0
                    padded *= 0
                    rir *= 0 
            return output

In [5]:
#IR Slicer (for BRIRs and RIRs) 
#You can adjust the duration of the fade out by changing the 'seconds' parameter.
def IRslicer(a, tail = 18000):
    seconds = 0.15
    fadeduration = int(seconds * 44100)
    fade_len = np.linspace(1, 0, fadeduration)
    if a.ndim == 2:
        mono = np.abs(a[:, 0] + a[:, 1])
    elif a.ndim == 1:
        mono = np.abs(a)
    maximum = np.max(mono)
    for i in range(len(a)):
        if mono[i] == maximum:
            start = i 
            break
        else:
            continue 
    y = a[start: start + tail]
    if a.ndim == 2:
        y[len(y) - fadeduration: len(y), 0] *= fade_len
        y[len(y) - fadeduration: len(y), 1] *= fade_len
    elif a.ndim == 1:
        y[len(y) - fadeduration: len(y)] *= fade_len
    return y

In [6]:
#Stereo Resampler - Resample the input signal if needed.
def resample2ch(x, originalsr):
    resampledstereoL = librosa.resample(x[:, 0], orig_sr=originalsr, target_sr=44100) #resampling the signal
    resampledstereoR = librosa.resample(x[:, 1], orig_sr=originalsr, target_sr=44100) #resampling the signal
    y = np.array([resampledstereoL, resampledstereoR]).transpose()
    return y

HRTF + RIR (stereo)

In [7]:
#Defining HRTFs, IRs, compensation filter:
filter_right, _ = sf.read(_HRIR[3])
filter_left, _ = sf.read(_HRIR[4])
ir_right, _ = sf.read(_RIR[1])
ir_left, _ = sf.read(_RIR[3])
ir_right = IRslicer(ir_right,)
ir_left = IRslicer(ir_left,)
compensate = loadmat('compensation_filter.mat')['hpcf'][0,0]['linPhase'].squeeze()
resampledstereo = resample2ch(stereo, 48000)
#Selecting stereomode
start = combinedbinauralprocessor()
stereomode = start.stereomode(filter_right, filter_left, ir_left, ir_right, compensate)
#Initializing processing
test = stereomode.binaural(resampledstereo,'HRTF')

In [8]:
Audio(test, rate = 44100)

BRIR (stereo)

In [9]:
#Defining BRIRs, filters:
filter_right, _ = sf.read(_NYU[1])
filter_left, _ = sf.read(_NYU[4])
filter_right = IRslicer(filter_right,)
filter_left = IRslicer(filter_left,)
compensate = loadmat('compensation_filter.mat')['hpcf'][0,0]['linPhase'].squeeze()
resampledcomp = librosa.resample(compensate, orig_sr=48000, target_sr=44100) #downsampled to 44100 if needed
resampledstereo = resample2ch(stereo, 48000)
#Selecting mode
start = combinedbinauralprocessor()
stereomode = start.stereomode(filter_right, filter_left, 'None', 'None', resampledcomp)
testbrir = stereomode.binaural(resampledstereo,"BRIR")

In [10]:
Audio(testbrir, rate = 44100)

SURROUND: HRTF + RIR QUICK TEST

(Playing signal sequentially across all channels)

In [11]:
#Defining sources
middle, _ = sf.read(_QUICK[3])
left, _ = sf.read(_QUICK[4])
right, _ = sf.read(_QUICK[2])
sl, _ = sf.read(_QUICK[5])
sr, _ = sf.read(_QUICK[0])
sub, _ = sf.read(_QUICK[1])
#Defining HRTFs
f_middle, _ = sf.read(_HRIR[0])
f_right, _ = sf.read(_HRIR[3])
f_left, _ = sf.read(_HRIR[4])
f_sl, _ = sf.read(_HRIR[5])
f_sr, _ = sf.read(_HRIR[2])
f_sub, _ = sf.read(_HRIR[6])
#Defining RIRs
ir_middle, _ = sf.read(_RIR[2])
ir_right, _ = sf.read(_RIR[1])
ir_left, _ = sf.read(_RIR[3])
ir_sl, _ = sf.read(_RIR[4])
ir_sr, _ = sf.read(_RIR[5])
ir_sub, _ = sf.read(_RIR[0])
ir_middle = IRslicer(ir_middle,)
ir_right = IRslicer(ir_right,)
ir_left = IRslicer(ir_left,)
ir_sl = IRslicer(ir_sl,)
ir_sr = IRslicer(ir_sr,)
ir_sub = IRslicer(ir_sub,)
#Compensation
compensate = loadmat('compensation_filter.mat')['hpcf'][0,0]['linPhase'].squeeze()
resampledcomp = librosa.resample(compensate, orig_sr=48000, target_sr=44100) 
#Selecting mode
start = combinedbinauralprocessor()
surroundmode = start.surround(f_middle, f_right, f_left, f_sr, f_sl, f_sub, ir_middle, ir_right, ir_left, ir_sr, ir_sl, ir_sub, resampledcomp)
surroundtestHRTF = surroundmode.binauralsurround(middle, right, left, sr, sl, sub, HRTForBRIR = 'HRTF')

In [12]:
Audio(surroundtestHRTF, rate = 44100)

SURROUND: BRIR QUICK TEST

In [13]:
#Defining sources
middle, _ = sf.read(_QUICK[3]) 
left, _ = sf.read(_QUICK[4])
right, _ = sf.read(_QUICK[2])
sl, _ = sf.read(_QUICK[5])
sr, _ = sf.read(_QUICK[0])
sub, _ = sf.read(_QUICK[1])
#Defining BRIRs
f_middle, _ = sf.read(_NYU[0])
f_right, _ = sf.read(_NYU[1])
f_left, _ = sf.read(_NYU[4])
f_sl, _ = sf.read(_NYU[2])
f_sr, _ = sf.read(_NYU[5])
f_sub, _ = sf.read(_NYU[3])
f_middle = IRslicer(f_middle,)
f_right = IRslicer(f_right,)
f_left = IRslicer(f_left,)
f_sl = IRslicer(f_sl,)
f_sr = IRslicer(f_sr,)
f_sub = IRslicer(f_sub,)
#Compensation
compensate = loadmat('compensation_filter.mat')['hpcf'][0,0]['linPhase'].squeeze()
resampledcomp = librosa.resample(compensate, orig_sr=48000, target_sr=44100) 
#Selecting mode
start = combinedbinauralprocessor()
surroundmode = start.surround(f_middle, f_right, f_left, f_sr, f_sl, f_sub, 'None', 'None', 'None', 'None', 'None', 'None', resampledcomp)
surroundtestBRIR = surroundmode.binauralsurround(middle, right, left, sr, sl, sub, HRTForBRIR = 'BRIR')

In [14]:
Audio(surroundtestBRIR, rate = 44100)